In [ ]:
import torch
from torch import nn
from .return_pred_architecture import ReturnPrediction
from ..diffusion import Diffusion
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

In [ ]:
#TODO: get x_data from diffusion/testing/data
# Put returns into y_data and drop them from x_data (no return columns should be in x_data)
x_data = pd.read_csv

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.2)

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32).view(-1, 1)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# diffusion has no labels
diffusion_train_loader = DataLoader(TensorDataset(X_train_tensor), batch_size=32, shuffle=True)
diffusion_test_loader = DataLoader(TensorDataset(X_test_tensor), batch_size=32, shuffle=False)

In [ ]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
from models.diffusion.train_controller import train_controller_diffusion
from models.diffusion.forward import noise_schedule
from models.diffusion.loss_func import loss_func_diffusion

# variables for diffusion model
EPOCHS_d = 200
lr = 1e-4
conv_hidden = (32, 64, 128, 64, 32)
mlp_hidden = (64, 32) # This is temporary until unett
input_dim = x_data.shape[1]
t_hidden_dim = 128
output_dim = input_dim
T = 1000
t_max = 1000

every_n_epochs = 20 # for how often to print epoch loss

model = Diffusion(input_dim=input_dim, mlp_hidden=mlp_hidden, conv_hidden=conv_hidden, t_hidden_dim=t_hidden_dim, output_dim=output_dim, use_conv=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

train_controller_diffusion(
    model,
    train_loader=diffusion_train_loader,
    epochs=EPOCHS_d,
    optimizer=optimizer,
    t_max=t_max, T=T,
    every_n_epochs=every_n_epochs,
    device=device,

    noise_schedule=noise_schedule,
    loss_func=loss_func_diffusion, 
)

In [ ]:
#TODO: add forward diffusion
#TODO: get scores